In [ ]:
import pandas as pd

In [ ]:
df2_translated = pd.read_csv("/kaggle/input/df2-sp-done-coreference-n/df2_sp_done_coreference.csv")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load Helsinki model
tokenizer_h = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-tatoeba-en-tr")
model_h = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-tatoeba-en-tr")

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

model_m2 = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer_m2 = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

In [ ]:
model_configs = [
  #  {"target_column": "model1", "tokenizer": tokenizer_h, "model": model_h, "src_lang": "eng", "tgt_lang": "tur"},
    {"target_column": "model2", "tokenizer": tokenizer_m2, "model": model_m2, "src_lang": "en", "tgt_lang": "tr"}
]

In [ ]:
# Function to fill a model column with translations for Helsinki
def fill_model_column_h(df, source_column, target_column, tokenizer, model, batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    sentences = df[source_column].tolist()
    translations = []

    # Batch translation for efficiency
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    
    # Fill the target column
    df[target_column] = translations
    return df

In [ ]:
def fill_model_column(df, source_column, target_column, src_lang, tgt_lang, tokenizer, model, batch_size=32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    tokenizer.src_lang = src_lang
    sentences = df[source_column].tolist()
    translations = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            generated_tokens = model.generate(
                **encoded,
                forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
                max_length=128
            )

        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        translations.extend(decoded)

    df[target_column] = translations
    return df

In [ ]:
# Fill model1 column for df2
df2_translated = fill_model_column_h(df2_translated, source_column="source_sentence", target_column="model1",
                            tokenizer=tokenizer_h, model=model_h,batch_size=32)

#df2.to_csv("data_nmt_2_with_model2.csv", index=False)

In [ ]:
for config in model_configs:
    print(f"Translating for column: {config['target_column']}")
    df2_translated = fill_model_column(
        df=df2_translated,
        source_column="source_sentence",
        target_column=config["target_column"],
        src_lang=config["src_lang"],
        tgt_lang=config["tgt_lang"],
        tokenizer=config["tokenizer"],
        model=config["model"],
        batch_size=32
    )

df2_translated.to_csv("data_nmt_2.csv", index=False)